In [ ]:
import time
SESSION_START=time.monotonic()
from pathlib import Path
import sys,os,json,subprocess,hashlib,zipfile
from google.colab import files
base=Path('/content/ns64_clean')
blocked=['/content/drive/MyDrive','/content/ns64','/content/imc_ns64_colab_migration.zip','/content/imc_ets_images.zip','/content/private.json']
assert not any(Path(p).exists() for p in blocked),'Use a freshly deleted and reconnected runtime without Drive'
subprocess.run(['nvidia-smi','--query-gpu=name,driver_version','--format=csv,noheader'],check=True)
bundle=Path('/content/imc_ns64_clean_runtime.zip')
if not bundle.exists(): files.upload()
assert hashlib.sha256(bundle.read_bytes()).hexdigest()=='39e9bb97ede83c777a06f385fbe4acb27fc4665ba1556b504dc64000dcafb925','Wrong runtime ZIP'
base.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle) as z:z.extractall(base)
bootstrap=base/'bootstrap'
subprocess.run([sys.executable,'-m','pip','install','--target',str(bootstrap),'uv==0.13.8'],check=True)
setup_env=os.environ.copy()
setup_env.update(PYTHONPATH=str(bootstrap),UV_PYTHON_INSTALL_DIR=str(base/'python'),UV_CACHE_DIR=str(base/'uv_cache'))
uv=[sys.executable,'-m','uv']
subprocess.run(uv+['python','install','3.11.11'],env=setup_env,check=True)
python=base/'venv/bin/python'
if not python.exists():subprocess.run(uv+['venv','--python','3.11.11','--seed',str(base/'venv')],env=setup_env,check=True)
subprocess.run([str(python),'-m','pip','install','setuptools==75.8.0','wheel==0.45.1'],check=True)

subprocess.run([str(python),'-m','pip','install','--no-deps','--extra-index-url','https://download.pytorch.org/whl/cu121','-r',str(base/'requirements.txt')],check=True)


In [ ]:
import urllib.request,shutil
manifest={'assets':json.loads((base/'assets.json').read_text())}
asset_dir=base/'weights'
asset_dir.mkdir(exist_ok=True)
def download(url,path,expected):
    path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():
        with path.open('rb') as handle: actual=hashlib.file_digest(handle,'sha256').hexdigest()
        assert actual==expected,path
        return
    partial=path.with_suffix(path.suffix+'.partial')
    print('Downloading',path.name,flush=True)
    digest=hashlib.sha256()
    with urllib.request.urlopen(url,timeout=120) as response,partial.open('wb') as output:
        while chunk:=response.read(4*1024**2):
            digest.update(chunk);output.write(chunk)
    assert digest.hexdigest()==expected,path
    partial.replace(path)
for key,a in manifest['assets'].items():
    if key=='dinov2':
        path=asset_dir/'dinov2'/a['filename']
        url='https://huggingface.co/facebook/dinov2-base/resolve/main/pytorch_model.bin'
        for name,h in a['sidecars'].items():
            download('https://huggingface.co/facebook/dinov2-base/resolve/main/'+name,path.parent/name,h)
    else:path=asset_dir/a['filename'];url=a['url']
    download(url,path,a['sha256'])
print('All seven asset hashes verified.')


In [ ]:
result=subprocess.run([str(python),'-I','-c',"import json, sys, os\nfrom pathlib import Path\nimport torch, pycolmap, asmk.hamming, curope, kornia, mast3r.model, dust3r.inference, yaml\nbase=Path('/content/ns64_clean')\nassert torch.__version__=='2.5.1+cu121' and torch.cuda.is_available() and pycolmap.__version__=='3.11.1'\nassets=json.loads((base/'assets.json').read_text())\nsource=base/'source'\naliases=yaml.safe_load((source/'conf/models.yaml').read_text())\nfor key,asset in assets.items():\n    path=base/'weights'/('dinov2' if key=='dinov2' else asset['filename'])\n    assert path.exists()\n    aliases[asset['alias']]={'kernel':str(path),'local':str(path)}\nalias_file=base/'aliases.json'\nalias_file.write_text(json.dumps(aliases))\nos.environ.update(DEFAULT_MODEL_LIST_PATH=str(alias_file),DEFAULT_EXTRA_DIR=str(base/'extra'),MPLBACKEND='Agg',HF_HUB_OFFLINE='1')\nsys.path.insert(0,str(source))\nfrom pipeline import create_pipeline\nfrom config import PipelineConfig\nconfig=PipelineConfig.load_config(source/'conf/pipeline/imc2025/mast3rhybrid/mast3rhybrid-022-c.yaml')\nprint(json.dumps(dict(status='DEPENDENCIES_PASS',python=sys.version,torch=torch.__version__,device=torch.cuda.get_device_name(0),config=config.type)))\n"],cwd=base,capture_output=True,text=True)
print(result.stdout)
print(result.stderr[-4000:])
result.check_returncode()


In [ ]:
EMBED='...'
KIT_PIN='4ce3bf6abbe6618112268988dde7109cb7b2cc187e27b4ffb503ed845dc037d9'
import base64,io,time
import hashlib,sys,zipfile
from pathlib import Path
from google.colab import files
raw=base64.b64decode(EMBED)
assert hashlib.sha256(raw).hexdigest()==KIT_PIN
kit=Path('/content/phase4i_kit')
with zipfile.ZipFile(io.BytesIO(raw)) as z:z.extractall(kit)
sys.path.insert(0,str(kit))
dataset='imc2024_dioscuri_baalshamin'  # Second independent group: fbk_vineyard
archive=Path('/content')/(dataset+'_opaque.zip')
if not archive.exists():files.upload()
assert archive.is_file(),archive
remaining=int(input('Seconds actually available in this free session now: '))
deadline=min(SESSION_START+14400,time.monotonic()+remaining)
root=Path('/content/phase5a3_runs')/dataset
ops=base64.b64decode('UEsDBBQAAAAIAAAAIQCQ7Tl2BQgAAHETAAAXAAAAc3JjL3J1bm5lcl81YTEvY29sYWIucHmlWG1z2zYS/u5fgcN9MFkzdNOkuTvN8IObKK2naeyxlVxvZA0HIiEJFUmwAChbzeS/dxcvJCXbd3NTjkcW8bLYl2efXYhSOtuIhryVFVsSo1ijW6kMYUp2TUnMhhPNak5uuqbhivAHrgqheUmWe/L2+hNRfK241kI2xHBtdEopPRG1lfGbls3JSsmatMxsKrEkfuIaXsMivemMqMKb4XW7EhV329JCNqBSYcLGd7cJaOrGEvKeGVYlZPrAC5Ahm4Tw3zscKTa82CagGysTUH/HE4JCc71hXrBy5nixzjg3pVWRNnLHq7yWJa/SlWhYlResKUXJDM93L8Mu7ew+OSn5Ch0DSvDICU60YWuelAlvygSN4I3J0OqInvtXGseTEwLPVpjMi0p/vpzZsTZzcnoHpK0dt/ZFYbUoQY4w++jdbZwcbwiTCYU32IXnkDBIYytuJRVpILpJCwiAP3RYBOvOqWhWXPGm4DlM6RQjSeNUQHh05PUe9Am+tVtRXowCE/vNLhUrYh3yt4yWHOADytNnhXj/nINVZzSXLfu94+kfogWHJe2cihoE5UwVG7HjdJFQtwLUbzuErR0npVgDGr2Z98JsemSlM47hY2r/TiheGKn2USlUcLiS0sSEaWIGBe+l2rrombgfLGRdAyj60FkDcz8aufjjxgQwWyZj6b2Iv5NryB2uQF/MM9C8KgFIBW8RzYQVSmptp1olCziHLDEpQXUS4WgtGwH6Q9gQj5rIptrDfohzAdCN08EAxdqWqwz1OafgXLVP2z09nk/vFUQ4N/zBRDSAfK8hkbqm3Sc2nxMi9V3zXFLfNbA+ZWq9y8KX+cvJ4g5PnNw1ThAmX46bo37Ntwv0UI6IyU7zvGaiyfPT+K5x3oAU752iCUdJNhxSp7zZCSWb+en1Txe309eX+bvp7ex0kZxab+ZcKaksfE/jsXk4kpZd3WoIfmEis295hh8R+C23iuR5AvkA6zJtFAzDgxYwYL+7hg5RRNP6F3wCInx4eBl5VMwnrxZnc5TmHR4vzsLUq8nCwcVxhsXPcIZ3w0zUXHZmikYdnjnKMKMEqyjB1JaGRCUmM6RGn8M6xzQYZ3F4rG0Ho94BTiYpgR4q0XCrDi95SQ/tBqLF45ZQRIqNOy+x3mU7JqC8VDx7zyrNg1/dvzh+UsoKtnSKj8VYCzOXWlsBuUenv07ffppdXn3Mb6Zvrz5Pby5++DClCcI/o7Oby4sP+ezyl+nVpxl99lDvW6gBBtxrfWtR9lA8cjHqdYyr/+5SuzqzvPrk3kcb4BS7ak4RjXSBtBzRCw00gfi36tGEfmZVx8PLz3zvvsYYddbso0di3bNFaV688wNdpJW85yqKbTHYuuM8eya0FrpmptjAV9HsgDiALeH7hmkcKsUKqkRfyJ6EkyvS0Yp+sXH76lnui/33FTS2VALOfhqNQ3H/f0UcgtlW8eooUUP5c0aH2ugR57ImhxLBRWvCoKebGpaGocDJ/v0Z18NDbVvSStGYfMeVWO1z61yug6hW8VUl1ptedtEpqMImt4aHQagZslNF2PaE9xGqluttCR7h0zVb0CS0ezvSL0rK86Fc47PhTJlsbnmykqzUEWa+Q4nlAPQZQNquW3LmVK4gGxDsjmTjVLeVMLgcuobFUDrRD9FSyiqy22MLNC2adV/ooOdsK65HCRJ4odODKx6xgt+WVbwJor/55suBf2jL2TanZ9tJzR4iJe8h0yRAVNPFfLuw9sEgmucEjPNCaY1Rsp8dfMZfRzwSKNgtlUqsEXOw0hEyNBXQTx/zY5xCvYa2+Zg7lj1nHKw+JqTlnPbcilyhiSVYe1Rg/wMife68sYf/OvN+vMqnHy5/vISh/Mfp1S/T2c1/egJeDtwTPzYoXXNoPnasEkCUD9hYgOeCXX9N555KUnz1ql79cDu9+XxhLZn+en11M1QKr4vTwrE2cF3s+n1IYq5WrICOH3q6JJRGbHw1CNjZJtc72fLTWso1dJ+FvWf5lglzUNslKLLotJHlPvoNshuQg6JGUcKczfxUih2tbVIi6ode0DNYf0aPgXKc8UE4DsQjegRF0lLeN5jrEbYodsGoStom3XIilKMVGI++x5TFTtRLBRKGCcu0cBdDDxEBfWmr0Fsluf3p4rvv3xCvKZkQgCMcJdrIXRiCLcET3gyYyvzB74XS0OR3ZgP59QeIZAb7emNb3wn5OP13Sq7wHgNhB/86FVDBiuHtdKQfrDHQZYKIcKjTbqxUuLuAAtBXgXA6JBZG/Zz64/XzyeW4DvkIw5llb15bGaBCVCBT0G9ffvfq9fdv/vHPf7FlATCglnDsHO4YnOL0G4W2azFYvMxc9NxrNMzDdZVlYdH8aaAMpOwhiaUdmvqjuPvRFO5noEOEkuN0wx9co3AcPnfmSFN0Vlpv4ZoVtQwLms5mquOJdVcut/ZtWP6Mc137vtxDJJ0GdoO7WWXuCu+SMfxCEB0mY+JhZROrzzrY76hiFDR/i3x0Y/d3UnpmN4AP7bV01Ek6dAW03gI5lf2V9DG+EFt+tkcO4cjfH2XDx4UlSD4kwHCphBDANcPfOb3AJGwZqVeBpAG8zuhn6gFkh9r3dnRwVwEnSvVilHp2SZ89iJBj27x+HBtHPX/xcjGncKjpoNRmmaVsuEI8YfDYthCdpGL1smREuwvS5Pj3FjccD0Rs1RuMRxIJ+lhC/5/usqwVhfiEn0mSVdXpzRFgn6BPv88zKCgDbkKjTv4EUEsDBBQAAAAIAAAAIQAn8F9aGhAAAHUwAAAaAAAAc3JjL3J1bm5lcl81YTEvY29udHJhY3QucHm9GtlypEbyXV+B62XpMMEca/tBYRwhz7S9E6sZzUryhB29CoKG6m7cNOAqaEmj1b9vHlVQHNLqaftBgjoys/LOpIQQF7VUSZNXZVJ46U6m+7rKy8ZLq7JRSdqE3llReFomhfaUbJK89Jqd9Daq+ipLTx6Tok2aSv1Ne3kmyyZv7r1NpQ5JEwohTmDZgUDJu6bI115+qCsFwItK5+X2xL7qo33cJXoHC+3rn7oq7TPA3DHAGp4caJ9xwjzrXdvkhX1r5KHe5IW072V7qO+9RHtlfcKgtErDsjrKIj5UmSzCTQ58iNOkzPIsaWR8fGOx1ImC0wGfAhg5tE2yLmScrV8KJtTt+pBrDQAsROAdzSJB9jlW1a0OvHcX5799/HT1YuC3ldpviurWgr66Pvt1eRWAxJIs8HRylIGHjIj1LuEnHedwEi0LmTbzaJQEwelGtSmeOt5Ima2TdO+wJC91DdsDEM1exmlV35+cfFleXn24+ORFnqhBmPL75E3Yq1X4Rpy8v8LJ/JC+ff32uzjLK522Ko/XCajYLjnkpThJi0Rr75ekSQr/C2iYXCpVqcXpiQe/fV5mCOHDp+vlr5cfrv+Ifzm7Pju3u5Z3Mm2RYv+yBX08zO5d/r5899s10DncewlHPoI5gGhftvty+e4CTnz28/lSnJxkcsMm5KNZSGI/KLDZnW+8siKpt5JH8KeSXEtzVLOcAcm/WhgCC4R/gSfvkNMyQ+HJIutB8gLvm6hbMg97Ix5o5+Npt9B7sE/fqMfAq9ZaqiMOM0wYFIaWLN9K3fhrgxb8QKtKa6shSO3t9z/AbLiTd2ap2Vi1TVodpK+bZIsqWBWZVD3xNAxq5PkiTWoAKkUgMpnmaCZi4ZyEMYp3Fx8/ny+vgdk4ulZJme4iVHKfQb8SPBai4wDqDR4eXInkmOQFSlfceDkqWKHlFMlvn86+nH04Z6HiBMv0CSDXCiQt8pIs2DOTeQGOcIqf1sTAdbCeZ0m4+PlqefnlDHXsKv4f9EyBDkniGW9TJFtD0ZSbRv/fGafPBKEAY3ATeRPHPniKTVCDxQeqLctOiPjDqbCOcHI4xisj/jecKpOD1NFqA34AVCB+yE9ff5c9hnW5FRg/vBy1Ag64lT7CXfE6LW4WN0NANupEWZ42PnmuSEw9pAg0+K8IXM4Pb+AXwDA4pyZ6fxV08GZ+taqaKq2KiImwrzGrvLgJ0PHCEXmaX4DGk45/O9BNqZh7bASsqAGEEyBcT3WczgFOCE0gMt7U8DxyuMrQIoZpzCwampuxtvkDdnwbcDEAl7/JtzxYrwS/9sfVVatS2U3za6zkkS32Zh4XZBgYMNEpd1tZ7s4MbDY8iSxvejbWSiJfYJk2mggx3+Hdbd7sKC0Iq1qWfilvi7yUkRALDK6b0wFZimQSQdYRvgeglyyizWKwiN0vLw3Jc5LKBiYyB6InyQhZDAFgII+KHHwhA+lnB7Hexz/9HKPVYLAy85VhE1g12oQim8DlQW9DA0I4esGGHuBg45AP7EYAiTEGcROBPXhgNZ7FHOcZjkIW4EP0dgha/XD65vXNYjFAb7XIQW+UGpH30jS+niWZOWJkijBK+tkrUVbxIalBRYw3D+UdsFP7gNRAAPB/tbmSGjM7qfIUIiErj0PCn9WaQwSAhGc3NPQct8kRLmIIYbYWiwA2rMwAHFpgAtTWvT30a72rf5zhWA8YwyP4OILAzOQRcTMvbZ4dytae8yAPa3AJu7yeUG4Crvh3KcI/IccycFYlK02Jsu9BAhPLFJykvzCInjDFDnOlhqo9QEpnq2qyS3EzA9rMOQxjiXp202IURRRocAlxTfrZRIloBerqGrTVz1A2AxfRbZ3oFSlmpwRWTceagOzaB0dkGDnhkXd7xusN3GS3rm7XYP87mcU8/5R7xN8aeLc7JGrf7e5G0O0OYpcjqTHKOVE+iROzPwUpbtmAmUcbUeofvotzXRXgmzKImfHD+6vH2KIVwUE2uyrDtbz0kOjm72p3v1aQf7x++zYVT+OCsASmAsDLGKWH3pkAVWqblPlXqfphQJzJY7xpiwJsPocaDkz+dMY5kxxX+5vgGOxdZUllXjvSLuu2ic3ovPGbSUjtVLrLj9KJdwO+mmlU46pO/mqhcjMjT0KcymoCdWR4BrI76kAvj93B4DlXVYnyGx+L/SgsAGUFGK1GJy7enS/PPsVfPsafoWz5x/LdP+PPZ1dXgh2+Abqu2jJLlLWNwX5aDZ6hkEnpffloOa0np2fEbCWQhQKBlbon7+ccfjxP3tRajNcPzwKHZFUC4h7arcy3uybGPAwB8eszUIwDI2AOI+ONkvKrnPVlzjL2/Okeqxezw/FEnYtiP0R+qtffQ9LYmgX43ZfGMYTufHMf07zU8yKlyalQSIT9HAZomBoC5EXNfS0tFJrPwdTrJFeYWmPtgL0fB9p4zU/R68AMpqScVgscUikZMy0e3+2VjIIr5WbZes60szUEe6zjpQ8JRrI9JEiY3Cpw2zGxAlzDRgKJSVFAQrDyRbUXwQLkcvWvc/AZ/XIxl9fNwAfvD0pTxns5j6EHreRGwiFSqWdhm3B+t3rNEfiOAoqDkJsuHsZjjzovprZZDGO/wUZvI0TEUB2Z/gucBgwV2zUqc7OLXm0ebPmEcR0SpQSf4KCkePhsdSQQzW0VH3N5G28lVBKNymH08ceIMa4E/wf7gN2UiRkqzfgcP3hqnElVKt9SlVZUW8zb0KTMo2ePMoSWgmuC4oBC8ww7c+JbsA8x13T5CrUk5kVed15v77VGNznFXQDrZ7gGLsZnpAtIgeHFSaPIRlAzSrCIPtHi5SH1eDQ6j47Lc7m5yWt4U+8/uLB3a8YsUFXVTEru2WSpT3anrRDyP+DWRoWisxvSvghxveoaMt0Uyy6CcdeKp2k2LZhJtME3R8+n0/1SVdUBpBf7qOu6+rwLWxCOlSwGWyLEGBZVkmmfHrP2UGsfpxwB274TeEhQb9C8oQNCxeSDMB+sor6aZ2evLrjwuW5MBhqbeRacLV2ysc0UMA8GEyHVuB2roU9VKUnjjG/R7YFOBd4ekiCMeCuoxVABV2bFdkFKucUxJmyrqraGpTcuI0gZokHHa65GROVfgxE8OM2vQHC7UjwGgj8WoGVCrPXasltldHmWX0MhgA/C7ni5dTnkNqBHMDaAAdSzU3bz/rR4zIKVwC4uxken2RtSZxcZ3K0yh4N16/5luKSPtPYIG/AIqNMzQZEIQW532/Ft1lv7KUvl00W8PP/w6weQSvzr8uLj8vryD47iqBNWUWhg7RIMhXM/i2n0PdCzhdIZolZGmTioE3gk0I+O10Zhh/Rag/AsRdeXH87O4+sPH5cXv10zKc+QQVwBj5ZkmNcDihTKGFT4QLQlCneLmT8ylLk2KzY8alLe+yDjfd8B4FqNO8d9AyJM9RGkILFlr8Oy/oohVbjqyCRBJup0lPCASoIhQbAaxxwwkeeEcTph17CDQbXHtHvRll0LHp0cpbuwcIb7HCFe6rpM2m8kgj6nkw4v7xEkbZaPaqSx6TydxrErfi5/I/BYw2JWBe7JyVRsz4RJmCTmZifsW+cZFHiwGXk0PTponpbeijLxXBEjxayTtzouXkKoeU82DTZyUVRGQjapdwketO47ltmPjuG1xM9kUEy9Bz9PrtEnljVTtTF9VfyS6TeL2fjaLaUvnCF+bUMsNi4yhDkNqovkPuq+0JmuRtChIm4iJgqteNopEIdXAQPElJH48nTO9pLwMtJv6pma3KZv+GLVMLTxOSRPfjI5ncSNlwcNcGjsGWNyKhjQ0eXYTyuZrJud6D+1QEz+U5po1cWKiFPWqSCdBj7Gn6j7cknhKCDH4x4rXv7++eLyWgTsY6MJcVPB2WBB/QLHLTrOyNLOn/+4pjN8GXe0h8IaurohK91W7SA3MMtmAiTr2Jo+stSVpoqAc6AZU2bjx7QIgi5fRei2Ocls0WLQ09HD47Afjh/K5zrigAoGIbKnsgRv8A1gbJsiBxBzrt5ADyE7grQ9aYvGd3djThZit7PMeNy0znv6OCWLbFI3zdw6FF1BMd86ntm67jO+RcBPg179K89mpM/0lofgD85XOEJ0MIjoq9sAEY3MQ2a1wdKJexEgcK5CR/2HftxWWFN0Bo8tsNryZW2I5+LX/6nK5O+fhrFlLzZmYF9QbkfnKWuqb0b2HAA7q9u4ztM9VA70TZkOlyiV3Ou5uGcw84KQLoQsAtN4sIox13noCKPJqVEkEYNclTdTg2HJhxkKnnwcmPSG08gEbxHUYOOm4ob9waJzX3DqHHi58ROwqJLCqHVsyY+vzVhPuM3mPPFt6fTl7He3aWVtvzwGeM9JZhGWbE61LYR4BxyWytNtXYM/0KYY1l5VQnZN2QK6tFbp/ChhxKDCcgZvS5lLGmh+4Pc0MB+YkeQl3ZGyWBp1P1t72TsSfKeHTk6fFe1X0oi/MvL06pT/h8QL3ry4MZ/pLG4anbZXDgcIKg+iu/lFrjwQeAvGPrcadtqXnYTifC0TXon1D38ABw3dwgvykpc+DhD1bQtANmliB6MWdDD6XBPMNfQhyrQKuRHT0ewgnHlTYCPYDoBiUPaju23Pt1+HhJvrJg+DxkrgJs/BtBgZxMVBbTIEThYMqsdi8P7j+Q/G5yAss5vuNRDgRzcsdjdmOCz2DEYwEzN8cLLLoG/UDCDa5g8D5PPO9sjQj2/AOGPKRXu7pDF9f4DKb2+GuWWDmmzOSs5kQ91R/Myjslz5i2F5ZJV/2sbHn13GKaO9amHsurtEMNjRfdWn5vtA0xdBd8mqm+k+ab8yHbMRBVqmXTLZ2wl+NsHexRPdWIp7sIAaNv4G/HkDBVKzsH33HfBug9duzCpq/MiUWu+ySGot6Z3gD8kh8+wIcox1TnQ07XQxmNe2r0lE8pL9MDLbQRuWbRkuk31ca+wh06PqH1utxx1Wo/QdOKGTQ02d5Z9e93YKjpPCAfjRMeN5ebRyWn7YZeDEhPoNSBVwYeylFiHyJ8Ybqf4i1ODOG1wOydXNjKoUsvQNrkUwJtUhlMcmOcjQUPSE20ispi/0FocDshl2i/Hn8Nvwl/4Sg6fBmI9gqCZJiG/3WE/fISlN1WDD4wZEOUeHORFugzg6KgpnXMVMUYtWae96ZCNDLIbX/7rMPjDl/cxxCJ5tjrt98RHkcVXeC8Fp6xKWYU833MrGHzZwF7YDSzmVoW1QKtnbdeDmJ71c/L2wJXCopzTjz5TvXK0f6jkj8jC9MPcTsMdCCQ3vm1/uNNZoY9+0C4GtqtFIc1/k2qxKHtEZplI4RaONPM7kLMqh3x3FtUXAciGSZ2a7+Ga48SwO9yJPHy4XQTdu0biTBicccn3fTDIj/M01JRwIE+OgPPIpi+z5YDOkRcA3TpyWwBMX9EZ35OhcOrI3usHMqMF2jx6RqHBrurtU1g3fBx61YfCi8Hjdkv4h20FBYWxmS3+3mGh+9Kxe9HnVKd4yTh/BkqhYgueT/wJQSwMEFAAAAAgAAAAhAKN6LaSECQAAsR0AABgAAABzcmMvcnVubmVyXzVhMS9ydW5uZXIucHm1WW1v27oV/p5fwXFfpFRRki636HqnAh2abBcbumItMKBBYDASHfNGJn1JKo5T9L/vHFKkKEvucgvMCGK9kOf1OW80pfRfG66ZFUqylmw128AtWSpNGDFC3rWcKLdA6YIYy+44YbIhS62euCT8Ed6JNZeWMPgjFq5LSukRvF+TDbOrVtwSsd4obclHuD3qr381SoZrZcKV5uHKrDor2nBn+XqzFG18i2zCddeJJlw/Cb/McS9rJa1mtQ38338qyKfP7/52Cd9XzLK2IJePvO5Q94LUK17fF4T/1uELzVkD+rIHXhAkuTAr5skaXZdSPfB2sVYNb8ulAMMtajCKaJjli4fzcqv0/bJV28DYcNYeHdUtM4b8u5OS6zdHBD4NX5LFQkhhF4vM8HYJfJWyIEsvOVx1xqpmV5BfVaeR0UbI3O/GD24qcU+Fxs3wKp++LNf3jdAZfxTGLtR99Vl3vNgwDW4z7mZvT2BfhYu9116mqv8ev+zlrCLzU8os+s+aEp1O43KxTJWqKvrh8j90UA0/zieZVAAtucsGfYTlGjXK84JKvk1hqMGBQnNDkOWOwCpeA3Z3NB9RdrT4g7PA9c30FexSusloDUCwvKEFpYVoYLmwu2iWMjwptPPq8MLfDyx5a/hYNYezLEArS22XF4ldCtqCAMaCjqA0PGx3oKNlQvIm2O87uiGOx8RnpEg2XJ/dFI2obQahbjtTDQZwsV/9kBkK2jMnYSXdQxwCYN4WRzFSepe4OPHSeZmK4+MGDdKaJDAAXCmZ0mHfZPnvcEIQbJAe9Ko7DWFT7wgkSMvWCDt5l2gTIHWc2HRk0FTyal/+AYgQnVaEMApKbIVdLUy3XIrHjJb9ktSUkK6y/nHheQ8vlQFQb1pWD0vmcZHCx38921ejdX16GC2JRk382iei6Ng7nnjJZwBfd4QM6Zt28l6qrfQFKbFASGpfv8VHWMrMsPf6jf8uhWz4o6ec34xh0STZy0xDK8L7gbUu52fg4iJu8e7uJcnH8dY/vTY3gwmbU4oFwmfHmUjW/A6wyzUH7BcDgWElpAMwbqCdBsywMdq2AJYT+2J25eUdt5kTheZVBctcoefX1EOW3kCKrtV603LLafIOHACvvIvQ2M5RCYjyYqQTIY4HWQuD/QVxRXUuufVlg6xACZ/Ao2YPEHbLXapVs18UZ5zkFyaO8mAM8HNISDC8Yv/TRxP/RAvvOwfbhyh+JzcMUT2Iz3S9EtBrYLppioGir86DansJq9+Wu310ZETMlWhEv4J8+vu7lz+9SkTHXBL6pfKL2FzBd6RHmCFP45hY8/Ut16Z6gsBZqhbky8ZYlWzNIfGJEinijQODQDD0e8dl1iMPloIdbCbLmhkOhJssdxslbnQ0c0AjLvM3DnjD7V+qVxf4aA9i+AFDZJqXy65t18zW0B3Ra3by9O7ky9nJnxc3yfXJzfF1mTy4eUELORFjjgXIgmwyUQbjLwC2+u3bl69z3JmdFa/ziSEOkTLdOvP2WxjxNDUgaHv+6vj87OXF8fGfMAkatuTRxUJiuE1bnSeUDUMBJW0OtHr/1zARaS568/0M97wIggYPNOUHQigJGAfzMD6Unzk25Ezv3oe2MIMGccj2Dvh2vxi41trmp32yG8sP6rvtIaan4Tz2hmUa8mxaYHDHaAmYy6860LXgJ2ncQjJ0nP3GHC0InaOqsWvBQNsoAc0xeHvZQnXaQ8i0O3WKg18Q+VlP87sOgQ4cJp05fzgdING7Jv+Dkmk669d4C4fkE982B60UrT7Svcl/n9t7hj2053xv15vg/X0Phc2HXeQTXKo+EYbgMINWwHTt35Avv3wkkvPGxDIIY8sdIJxOE4XVuykn58Oe2nfQmIpSgGqn1JuPpgXnigEYpnz5Y82hFl+6L5iX54X4I/kIkck1ZKPbHUwtPxNGtlqBjlEzNAOwUMmg9rgBTAoc3R4E35azlIO5e0iGW9cRj57gzQtaQnhoBqOG5Cf0BR4RlPjvIsvLFX/M5xIw4JoJww9EWrCdD6zoOpc1fzjC4qFB5bwxkCmhMNPJ8tmSHWkUdEuL+PaXj4v3l1f/fPf58n2BTRsY2rTQj7XV+Vx1Dx+sOEvXv0E0QzpuklF7fgd+nsqthnXZslg6+8/AZ0hXI2gO0scaEh/lI4zmwfL+oEh1UC+tBjNNmcGk0wp5n5hmklr6FDZppJIK08k0m3F3UAQ5BuZpIM4hZCAU53LaM2fKaXcbCIQZZqbsDmu0UH3d6uf2k/OhvfKZx60Zuvc/VHip/WFGJwEQqn2ADhHgxrXuNnZouX8mfRQ37qSNwMDbYQMCW0dFfY8DzAdLGGNh1dwpzmCwKljBNT0DkWSQcI4OWQH2EdbZldLiyR1UhtyRijM+KppOJsgqOzTMKD2DbW9zWAcwg5loZhoK9szzA4MP7Xm3eAyzC+YFs8LqyH1Oh2yv8uWxziTJxxOPebTvifxwJGpnqbG/xgrNOykcF/mjOzD5HZ5vnt5CNq1XfWN3TV3eW/hgpDdY19yZYlxP1C0CyMlg5tyVSNPwWhiU9TkCOdHH0vi5dSRRHiqtq2aF35XK1PcrpMdr0s5uwHMVNuJxHoaU3hhanB1wctwq1bZy599rBayVFHWWJpw1TLVYn9dCZucXF2dnJ45XzCgnsH8fCnHX2/OXZ2cew85sEMw120AOc+AdqL+tXgNlAImQ7pyoFngmaiD5o9K3XQM6padVsqmA7QuUKdI4QV7FT0BnihlXyAZ2k0nYH5rGPONjOSSTCtihlvHoOZyRDFT8xsNmTFvCLOTQuQo/7OgPv4cnkxaqz+1x8gEx96s325gZsU56eccTAR7CwSDkFjuE9ud/HkhVT2yvB5nt6wKpOKR5cuPBbQX44ToKP3/u1Ldv2T/47lJrpQvsHT4oe4V11D1xXQEsm0rh2iL/g0m2pF8do28xponq7Kazb8hX2PwNIq8/z6kPnTOPK1qQt6DQokAmAKQqbQD+TN4B8abD89XeQ2L+fOw5I2zggtPpfJOLb7ImnJGN6Mbj7Qlef0SG8e44ZO0n/CFGvev+ygyP3fesr+4FRDPENx5AQL9aFxSf0CL+xlXi/WSQoX7WwPM4d0xCXYLzJPKyVVsA10zv57g5UDiyh39B6VNstKyPgkOhVDi6+A/SBgO4V0EUiCJ2JxWUvtpUzUyfCbq4zYlULmNiHYACI4GDrLmzjFsxo1QK9cD3AKLHw8L4N6NQ2Z+rM2Jv70h9QMLoSCU/+i9QSwMEFAAAAAgAAAAhAC10uak2CAAAGRQAABcAAABzcmMvcnVubmVyXzVhMy9jb2xhYi5weZVYW3PbNhZ+96/AYh9MxgxdN5duNcMHryNPPU1jT6y0OyNrOBAJSagoggVA2Wom/73n4EJSsry75WQcEpeDc/nOdw5EKZ2sRE2uZMXmxChW60YqQ5iSbV0Ss+JEsw0nn9u65orwJ64KoXlJ5jtydfeFKL5UXGsha2K4NjqllJ6IjZXxu5b1yULJDWmYWVViTvzEHXyGRXrVGlGFL8M3zUJU3G1LGyW3vGZ1wcPWK1mDkoVJSCE3GwYqMk2WHHRjhud+LCGfx1fjm7vJvZOjVZEqa0D+jl2khZcRZH64T8g1M6xKyPiJF6CPrBPC/2hxpFjxYp2AnQzEarblCUEFc71iR4W71yDaua1fWIM9Vb6RJa/ShahZlRegryhR+e1F2KWdR09OSr5Al4NKPHKCE23YkidlwsFMNITXJkN/RvTcf9I4Hp0QeNbCZF5U+vPNxI41mZPTOSFt7Li1NgqrRQlyhNlFH+7j5HBDmEwofMEuPIeEQRpbcQupSA24SRrAFvxD90Ww7pyKesEVh4jmMKVTxAiNUwGB15HXu9cneNpuRXkxCkzsm10qFsQ65B8ZLTkAE5SnLwrx/jkHq85oLhv2R8vTP0UDDkuaKRUbEJQzVazEltNZQt0KUL9pMSHsOCnFEnDuzXwUZtVhNp1wDB9Tuw9C8cJItYtKoYLDlZQmRrSaXsFHqdYueibuBj2Is0NUh+h4COBewMFA+mGkepH/JHeQpVyB/pjRYElVArAK3iDWCSuU1NpOQcYVcAyZY/qDKSTC0Y2sBdgDYUR8aiLragf7Ie4FQDlOe4MUaxquMlTunIKz1S5tdvRwPn1UEPHc8CcT0QD6nYY0a+tml1jmSIjUD/VL9PFQw/qUqeU2Cy/Ti9HsAU8cPdROECZjjpujbs13M/RSjgjKTvN8w0Sd56fxQ+28AQTQOUUTjpJseKROeb0VStbT07ufLu/Hb2/yD+P7yeksObXezLlSUlk4n8ZD83AkLdtNowEMhYnMruEZ/onAb7lVJM8TyA9Yl2mjYBgetIABzz7UtI8imtZ94BPS1YeHl5GHynT0ZnY2RWne4fHsLEy9Gc0CdpBDLJj6M7wbJmLDZWvGaNT+mYOMM0qwihJMdWlIVGJyQ6p0Oa1zTIthVofH2rY36h3gZJIS6KISNbfq8JKXdN9uoGE8bg7lqli58xLrXbZlAgpZxbNrVmke/Or+i+OjUhawpVV8KMZamLk8WwvIRTr+z/jqy+Tm9lMOpeX21/Hny39/HNME4Z/Ryeeby4/55OaX8e2XCX3xUO9bqAkG3Gt9a1H2VDxzMep1iKv/7lK7OrM8e3Tvsw1wil01pYhGOkOajqitgzShlxroAvPAqgkDv7Kq5eHjZ74Lr9egzydprpEv3FiMiGD1Lnp2pHvWeJI/2vmIztJKPnIVxbZwrJ0qnmkTuhF6w0yxgldRQz+AzArvK6ZxqBQLqChd0TsKNVfeowX9amP6zTPgV/vfN9DY0gwE4jhS+7bg74rYB7qt+NVBEodS6YwOddSj0WVUDuWEi8aEQU9FG1gahgJf++8XXA8PtQ1NI0Vt8i1XYrHLrXO5DqIaxReVWK462UWroGKb3BoeBqGeyFYVYdsR7yOMbR2w5XqAXdfyQZlqdnakW5SU531px2fFmTLZ1HJoJVmpI2QFhxLLD+gzgLtdN+fMqVxBpmAiOAKOU91UwuBy6DBmfZlFP0RzKavIbo8t0LSol10RhM63qbgeJE/gjFb3rnjGGH5bVvE6iH716uuef2jD2TqnZ+vRhj1FSj5CFkqAqKaz6Xpm7YNBNM8JGOaF0hqjZP+28Df+NuCYQM9uqVRiiZiDlY6s45exF1rmY9SCnjoaS4IdeFQeDveudFni8Uvo2X58w3PQqHl5STfQH42GwB0FMh7K3VF5z6okPoc9rKscUWimhy3a3l5P2ZY+/p8y5tdHt/eWC5NAlElPn8kE6Na+xUfZvxP8jLSA/FglSjL0AN8iYxR8RL6CpD0esrcHSHOuFqyA+wN2iKGwYhutoVBtbcvsUWF3LqVcQi9b2Pugb7gwDtouQZFFq40sd9HvkP+ALRQ1gBVmdeanUuyPbYsTUT/0mp7B+jN6WJUOOSEIx4F4QKCgSFrKxxrZIMIGxy4Y1FiLJMuaEb1cgPGYs4hE7GO9VHAPTFguhnseeogI6Gobhd4qyf1Pl9+/e0+8pmREgE/gKNFE7voRbAme8GbAVOYPvhZKw5WhNSvIwD9BJDN4SzC2cR6RT+PfUnKLtyKILPjXqYAKVgxv0QP9YI2BHhVEhEOddkOlwk0IFICuDIRTm5a2KcOon1N/fHfdgtYd7urDNsLlODIWhjPL3r+1MkCFqEB2oN9dfP/m7bv3P/zrRzYvAAbUkoedwx29U5x+g9C2DQaLl5mLnvuM+nm4/LIsLJoeB0pP2x6SWPzhSnAQdz+aAmeADhFKjtMVf3KtxGH43JkDTdFZ6WYNl7aoYVjydDZRLU+su3K5tl/98hec65r/+Q4i6TQ46Rkocz8IuGQMv2VE+8mYeFjZxOqyDvY7uhoEzd9Jn93//Q2XntkN4EN7yR30oQ5dAa33UNTK7oL7HF+ILT/bIYdwaK/JJ1nzYekJkvcZLbArhAAuKZ50vcAkbBmoV4GkHrzO6COQtaI50H1nRws3HXCiVK8HqWeXdNmDCDm0zevHsbXU09cXsymFQ00LxTjL7PUALiBHDB7aFqKTVGwzLxnR7no1Ovz1xg3HPRFb9XrjkUSCPk+I9P/pLstaUYhP+NElWVStXh0A9gh9+n2eQUEZcBMadfIXUEsDBBQAAAAIAAAAIQCxxt5GYwcAAN0RAAAcAAAAc3JjL3J1bm5lcl81YTMvcHJvdmVuYW5jZS5weZ1Yz1PcOhK+81dotRd71/E8yCYHqnwgMC9LbRJSMEm9rQnl0tiaGT1s2U+SJxCK/3279cueAS5LDiC51d3q/vrrViil51vR1G961dVDxWuyUkxWW8J3ouay4jm55tp0ihPFq67tB8M1MVtONGs5WQlZC7nRZK26lmjOGtDQM8Wl0Tml9Ei0facM+VN3Mvzd6SMr3TOzbcSK+O2vsAwihrf9WjTcCWpV5WqQkqvyHTvOq04axSoTDp6HNdPkA9M8Ixc3GblZnH2cw+9qy6u7jPC/BtZkcAdWZ+D6DqTQQKm3LCO12MAdM9INBq4In+AGRhgBPh/dnP97/vmsoP0WVL9jb3MXn/yYHl3Pz+eXXxc3RUK53AnVyRbuneNdaUZX3SBrph7CWkiIXQlB5KKPQhhjsOSF0qOrb4uv36xGKx51OZt+1Steiwrdyyu9gw2uVKd0LvtfsFgz0QyKB41HNV+TislOioo1yY41A09Pjwj8KG4GJf3tE5TP66HttRPKNAS3vOMPulgoXHKICgMkaPAug3+nNE1zgEhX8yQFQ1XDtI7ZSDAV3hC6sIXIc5Vo3qyzfzC10f7bniNwTA89iKW5l3ei2Y4rDfedpMHmte8EBPwYrxnseEQ6Q9qwDc9U15mJNQfPAndntGI92OYzt5nXKzrKCVkEjCTuexo/WjwlIJIhpJJ9ZX92Kx/+JXUH6e2Srlh1N/So7eTde3qbUTm0XGFWvEt0VN8yfVdEFHrrGZrDe+USak+ny+NbItbEXrIoqFGCNZTwRnOyvH05uBC1lhUO0y46hYtRzQwE1hRQOq7UCmvI/Z0BOZiu6hq3GUALl/P7cJtob/rzs1N3QdV4yu3CGYBlLcAyL6Gm12LjBPsldcsxVD5leP8X7WBFQGFpjeh81KcxbTYvekaRmXxKyLpTRBMhPUUsT93vHIDD7xMbjvT26TVDXd9pXpc2QWNZ4TKdotAW7BSE9SEMA9CD6FQqCkEhAc4Ki7F6dkgXS08MJb/n1YBIoWPe+X3PK8NrF9RYFmNFREkjWg7UV6CFffbI+b3QRicpgUyR4MWBzIabhCIL0BRQuLi+PPtULi4/z4HL6EG9uOtkFovBP4yPGXRBIQ3Xi/kFZAhA7X1yaDZctULC+ekZjFYaqJHECBDPsIH64lkfgXjUZYINtTBjfEfStV4c7o3xsH596STfZ1K418QOXsv3lCR63LIeEl823QaR4zE+gZL1aEnZ2mCNWDfs1mjyOTTBCDgJ0Jcj9OuZdFiXiHXfWPylZJoLXaJokj75QEGplxWOAgBm5JkIFAAlViz0g0FVHElIA+ntBBIy0BEPoYy9WnY73pQtIKLJ1xj7ciz03XFo2xrKFTssHg0tvYjdY8+Kg6qDkD8W6SS5uEmzcD7uZtRexROQRW/kqqm2GK2g9j+Xi5mnHz2bXGT2/CL5A2tBmw3Ic8ryDsQDxAkM0EKFbczTKwGiT347eX8MPxl1RUJw9xVfbSKcm7Ne9LwRks9EW4GOdzOgIvNWbR9WStR7ize/nZy8qfa97odVI/QWCM25h443Yhe89fY9pIsY5tfYJIS/cN3G+lmGLBbO5MEutoFp6IpXHYuoX8HQsW2ZuvOycQ1CGLXCBhSKrB2MjXZQbcscITF7aRrDZv3szNQuso8SOOOVoi7WVOr3/yqF7hpIbw2AKB8vbp7KR7T+RLOWm21Xo6QTnKaihFRUdNSMgzVToEqWK2jDmE97sFMbJsUvoIu4DWZqvivXQ9PQ7JD+i5cYaCRWlyScuwsfhb1+4koR56pEdsbKRcLLKGhreHgdhIMeIDhQJyifhQ/pXl3nUHw7LuEot/F3X5FnEo+wSMHTUdeDak3QnRVw4g46D1s1nN5O+ijaPuxKrsXcQSAKOv9jfv5tcXn1pYSB/er7/Prsw6c53Zt+sHsV9MtVOf90+fESPpcf51ef54vr/1Kc70BjAebdX8DJviob8Gzl2h9MzJBVAAgwG7YOTX5nQNf/l5e/ny3OPr3k39WHm/n19zMrNP/j69X1YuqdM17at0D08e/kO3qG/DO+2fYfeFBOa3zewWjfYgNqHogvPsBQbJ/EgSm3Wn8Ks41vtHzBkdDhpXMhoJ7ggfCQ1EJZhKX4JjNjFHaC/yzwoZdMxg/bo9AxaFPhSXW61+I6nQP271zpomiGiuxfL6vx3W5fi299eCjC+vR5K/W26lcN4cMmTG0B4L5dovTBcDWOvc9nteKV4QTVvF5Y7kXXtS20ltC3/EHbp90MGXtiGHTUZleELmely6hjPAvj/3QEDa8LjGkCbCQ26C28/dxbYzrNVgqI1OUWTS3f3qYTegP4Ookc67w0/N4kk++Wc1AK3pO1Rngl9Id0/PADeCCjgwyzH84Zv7gkrr9CUNQDPTBlNSneAufoYb0W9wkdde2J/rOgz/+L4e2ErsK0EqejH5IeXDr/qQQMBPZOVueaHsxSjzaIf1NPmY0PLJV9maSwlU72arcxnbYeRwrF8+PSeYt7dgSLN/OjKObg6H9QSwECFAAUAAAACAAAACEAkO05dgUIAABxEwAAFwAAAAAAAAAAAAAAgAEAAAAAc3JjL3J1bm5lcl81YTEvY29sYWIucHlQSwECFAAUAAAACAAAACEAJ/BfWhoQAAB1MAAAGgAAAAAAAAAAAAAAgAE6CAAAc3JjL3J1bm5lcl81YTEvY29udHJhY3QucHlQSwECFAAUAAAACAAAACEAo3otpIQJAACxHQAAGAAAAAAAAAAAAAAAgAGMGAAAc3JjL3J1bm5lcl81YTEvcnVubmVyLnB5UEsBAhQAFAAAAAgAAAAhAC10uak2CAAAGRQAABcAAAAAAAAAAAAAAIABRiIAAHNyYy9ydW5uZXJfNWEzL2NvbGFiLnB5UEsBAhQAFAAAAAgAAAAhALHG3kZjBwAA3REAABwAAAAAAAAAAAAAAIABsSoAAHNyYy9ydW5uZXJfNWEzL3Byb3ZlbmFuY2UucHlQSwUGAAAAAAUABQBiAQAATjIAAAAA')
assert hashlib.sha256(ops).hexdigest()=='dbefffec1ede534a31bad1f8cd84da7b61f484e125ef6ef1adeb5d4698ecd886'
with zipfile.ZipFile(io.BytesIO(ops)) as z:z.extractall(kit)
from src.runner_5a3.colab import interface
run=interface(root,deadline,{'images': 138, 'dataset': 'imc2024_dioscuri_baalshamin', 'worker': '5429666b9249c2bf685615d95c596b56b650293bd1ccede9178334e243cd8ffa', 'protocol_sha256': 'ec1af922c449ed0f68cc321cf972af772e491154ddaa36700a75632a24c4a690', 'config_sha256': '8134dd74471fe6bd25bf8ce3a891454f461e772d156715409f62977eb4c95b26', 'source_revision': '07b5695b99a8ae4725d5d952f0065c2157fadd7d', 'image_permutation': '1b7bc97f6f54a2f5daa967a06e67211cd565f84d2d9403cd60f1faa50af543b0', 'published_config': 'e7829be60865192d3ddcc8555b94bc8678415e265a566d8ce8386ef17b0cc763', 'benchmark': {'benchmark_hash': 'a78f0542413b8ac2ec8c262988ed5303c934490a2b31d2a45d3939dd6d3ab2a4', 'evaluator_hash': 'c7ae94306ba0ae4ce9c00e9abc7e15bd99f3091fa063b5826b0dd567a6c3f261', 'split_hash': '170546743202f21b9bcc98376ff09a6eae2da16ce833115478a976c4f46b2057', 'threshold_hash': 'c6aec2fc2c8f9cb5f268f57333911f0835fcaba8aedc0eec67951ed729e68645'}, 'image_archive': '2390cb5c85812982edfd47d73471e548fb70e5e8f4d4d76b35674951f089fd9c', 'options_sha256': '34efa9cc0082dfefd16df98fea3ad8e9523d75245f464072c78985a24a7bb7be', 'source_inventory': 'edb517b23a85fffe9c4419bed989ca121ea0e8845693ff788614e13cbd713ddd', 'environment': '8bb71679bf5735f6e3c8225885be88c1f530639094df310a5a29e6691498455b', 'weight_pins': {'superpoint': '52b6708629640ca883673b5d5c097c4ddad37d8048b33f09c8ca0d69db12c40e', 'aliked': '5be8704840ed662d9d8c561bf7279c222092674e7eb05fd0feab94899e9d82f2', 'mast3r': 'e28f91b488554653e2b46ddae9c78c1143e0bcb2e27d3e26cdb0b717f1568eb2', 'retrieval': '000a324f06f9369ab3e32d03dd4b8a838d48e98e3f3bf9d7dd034dce1bfd4c50', 'codebook': '700af9e31ad0ca6159955dc80b4fd6e14234780b52f5c91884f6553991933c1a', 'isc': '512390a15ab52f8ab5adc14a315b157a5d6e732e3361c7a0dce5be6e95a86d30', 'dinov2': '014965d9e330e7f4bff8ddcbee9df5e4f2ca032b2f5180942a6edb454783e75d'}},'dbefffec1ede534a31bad1f8cd84da7b61f484e125ef6ef1adeb5d4698ecd886')


In [ ]:
run('capture')

In [ ]:
run('original')

In [ ]:
run('trial')

In [ ]:
run('decision')